# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yowxy/Flyrank-Internship-ML/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane: CTR / Engagement Opportunity Scoring**

I'm choosing this lane because the starter data already shows a clear, checkable pattern: click-through
rate collapses sharply as position drops, and a meaningful share of pages sit in "good position, low
CTR" territory — pages that are winning visibility but leaving clicks on the table. That's a concrete,
actionable gap: unlike a raw decline, "good position + weak CTR" points to a specific, fixable lever
(title, meta description, snippet structure) rather than a vague "something is wrong." It also builds
directly on Discovery B from Week 1, so I already have a working intuition for the signal before
committing seven weeks to it.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, pandas as pd

# Find repo root from wherever this notebook runs (Colab, VS Code, terminal)
while not os.path.isdir("data/raw") and os.getcwd() != "/":
    os.chdir("..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("Working dir:", os.getcwd())
print("Rows:", df.shape[0])

visible = df[df["impressions_90d"] >= 100]
ctr_by_tier = visible.groupby("position_tier")["ctr"].mean().sort_values(ascending=False)
print("\nMean CTR by position tier (impressions_90d >= 100):")
print(ctr_by_tier.round(4))


Working dir: /home/iklil/Internship/FLyrank/Assingnment1/flyrank-ml-internship-starter
Rows: 30000

Mean CTR by position tier (impressions_90d >= 100):
position_tier
page_1      0.3548
top_3       0.3341
striking    0.2558
page_3_5    0.1424
deep        0.0554
Name: ctr, dtype: float64


## 2. The question: decision, action, cost of a wrong call

**Research question:** Which visible pages (enough impressions, decent average position) are
under-capturing clicks relative to what pages in the same position tier typically get — and are
therefore worth a title/meta/snippet review?

**Unit of analysis:** one content page (`content_id`), evaluated over a fixed 90-day window.

**Decision improved:** which pages a content reviewer looks at *first* when they have limited time
to run CTR-improvement experiments (rewriting titles, meta descriptions, or snippets).

**Action someone takes:** a reviewer opens the flagged page, checks its title/meta/snippet against
what's ranking well in the same position tier, and decides whether to rewrite it.

**Cost of a wrong recommendation:** low-to-moderate. If a flagged page turns out fine, the reviewer
spends a few minutes checking it and moves on — no user-facing harm, no site risk. The real cost is
opportunity cost: reviewer time spent on a false positive is time not spent on a page that truly
needed the fix. This is why precision at the top of the queue (Precision@K) matters more here than
overall accuracy.

**Why data/ML helps:** CTR expectations differ hugely by position (page 1 gets roughly 6x the CTR of
"deep" pages), so a single global CTR threshold is misleading — it would flag mostly low-position
pages and miss real under-performers near the top. A model or a position-tier-adjusted score can rank
pages by how far they sit *below their own tier's expectation*, which a flat rule can't do well.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# No new query needed here — this section is framing, backed by the numbers already
# computed in Section 1 (the CTR-by-tier table) and Section 3 (the opportunity pool size).
print("See Section 1 and Section 3 for the supporting numbers behind this framing.")

See Section 1 and Section 3 for the supporting numbers behind this framing.


## 3. Quick look at the data (2-3 real numbers)

Two things stand out. First, CTR really does collapse with position — page-1 pages get roughly
6x the CTR of "deep" pages — so any opportunity score must be tier-adjusted, not a flat CTR cutoff.
Second, nearly a third of all pages (32.5%) sit in a plausible review zone: decent position, real
volume, but weak CTR. That's a large enough — but not overwhelming — candidate pool for a ranked
queue a reviewer could realistically work through. The weak raw correlation (-0.073) confirms that
position alone is not a strong enough single-variable signal; this needs a model or a tier-adjusted
score, not just sorting by position.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# 1. Size of the opportunity pool: good position, weak CTR, real volume
low_ctr_good_pos = ((df["impressions_90d"] >= 500) &
                     (df["avg_position"] > 0) & (df["avg_position"] <= 20) &
                     (df["ctr"] < 0.5)).sum()
print(f"Pages with position 1-20, impressions>=500, but CTR<0.5: {low_ctr_good_pos} "
      f"({low_ctr_good_pos/len(df)*100:.1f}% of all pages)")

# 2. Position and CTR move together, but weakly and noisily on their own —
#    the model needs more than a flat correlation to be useful
corr = df["avg_position"].corr(df["ctr"])
print(f"Correlation between avg_position and ctr (whole dataset, unadjusted): {corr:.3f}")

Pages with position 1-20, impressions>=500, but CTR<0.5: 9759 (32.5% of all pages)
Correlation between avg_position and ctr (whole dataset, unadjusted): -0.073


## 4. Careful words: what I can and can't claim

**What I can claim:** which pages, on this dataset and in this time window, sit furthest below the
average CTR of their own position tier — an observed, relative gap. I can rank pages by that gap and
say the ranking is directional: "these pages look most worth a title/meta review first."

**What I can't claim:** that a low CTR is caused by a bad title or meta description (I have no
experiment — could be intent mismatch, a stronger competing result, a SERP feature stealing the
click, or something else entirely). I can't claim this reflects Google's algorithm, that fixing a
flagged page will recover clicks, or that my ranking proves anything about causality. Every result
here is observational and decision-support only — a prioritized list for a human to check, not a
guarantee.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# No new computation needed — this section is a written boundary on the claims above,
# not a new number.
print("No additional code needed for this section.")

No additional code needed for this section.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.